In [0]:
import os
import sys


SRC_PATH = os.path.abspath(
    "../src"
)

if SRC_PATH not in sys.path:
    sys.path.insert(
        0,
        SRC_PATH
    )

print(
    f"Project source path: {SRC_PATH}"
)

from sample_assignemnt.ingestion import (
    RawDataIngestion
)



In [0]:
CATALOG_NAME = "retail_sales"

VOLUME_PATH = (
    "/Volumes/retail_sales/raw/"
)

print("=" * 70)
print("RETAIL SALES RAW INGESTION")
print("=" * 70)
print(f"Catalog     : {CATALOG_NAME}")
print(f"Volume path : {VOLUME_PATH}")
print("=" * 70)

In [0]:
ingestion_pipeline = RawDataIngestion(
    spark=spark,
    volume_path=VOLUME_PATH,
    catalog=CATALOG_NAME,
)

ingestion_result = (
    ingestion_pipeline.run()
)

In [0]:
summary_rows = [
    (
        source_name,
        source_result["table"],
        source_result["rows_written"],
        source_result["status"],
    )
    for source_name, source_result
    in ingestion_result.items()
]

summary_df = spark.createDataFrame(
    summary_rows,
    [
        "source_name",
        "target_table",
        "rows_written",
        "status",
    ],
)

display(
    summary_df
)


In [0]:
failed_sources = [
    source_name
    for source_name, source_result
    in ingestion_result.items()
    if source_result["status"] != "SUCCESS"
]

if failed_sources:

    raise RuntimeError(
        "Raw ingestion failed for: "
        + ", ".join(failed_sources)
    )

print(
    "All source files were ingested successfully."
)


In [0]:
expected_raw_tables = [
    f"{CATALOG_NAME}.raw.customers",
    f"{CATALOG_NAME}.raw.products",
    f"{CATALOG_NAME}.raw.orders",
]

missing_tables = [
    table_name
    for table_name in expected_raw_tables
    if not spark.catalog.tableExists(
        table_name
    )
]

if missing_tables:

    raise RuntimeError(
        "The following Raw tables are missing: "
        + ", ".join(missing_tables)
    )

print(
    "All expected Raw tables exist."
)

In [0]:
display(
    spark.sql(
        f"""
        SHOW TABLES IN {CATALOG_NAME}.raw
        """
    )
)

In [0]:
print()
print("=" * 70)
print("RAW INGESTION PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 70)

for source_name, source_result in (
    ingestion_result.items()
):

    print(
        f"{source_name:<12} | "
        f"{source_result['rows_written']:>6} rows | "
        f"{source_result['table']}"
    )

print("=" * 70)